## Prepare bølgemodel data

In [2]:
import waves

In [3]:
waves.prepare.bolge_model_data()

Rasterizing aoi mask...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:51.
AOI mask ready: /home/kim/work/eswm-bolgemodell/niva/tmp_outline_mask.tif
Copying raster for filling...
Filling nodata (0) inside outline...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:27:40.
Coarse fill: downsampling 20× ...
Coarse fill: FillNodata at low resolution...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:11.
Coarse fill: upsampling back to original resolution...
Coarse fill: merging into raster (remaining nodata only)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:26.
Clipping raster to outline...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:44.
COG saved: /home/kim/work/eswm-bolgemodell/niva/EswmRaster_clipped_cog.tif


## NiN 3 – bølgeeksponering (SWM-klassifisering)

Mapping of SWM (Shore Wave Model) raster values to NiN 3 wave-exposure classes.

| Klasse | Trinn | Trinn-navn (NO) | English name | SWM-grenseverdier |
|--------|-------|-----------------|--------------|-------------------|
| 1 | 0 | minimal vannforstyrrelsesintensitet | still water | < 1 200 |
| 2 | a | svært beskyttet | very sheltered | ≥ 1 200, < 4 000 |
| 3 | b | temmelig beskyttet | moderately sheltered | ≥ 4 000, < 10 000 |
| 4 | c | litt beskyttet | slightly sheltered | ≥ 10 000, < 50 000 |
| 5 | d | svakt eksponert | weakly sheltered | ≥ 50 000, < 100 000 |
| 6 | e | litt eksponert | slightly exposed | ≥ 100 000, < 500 000 |
| 7 | f | temmelig eksponert | moderately exposed | ≥ 500 000, < 1 000 000 |
| 8 | g | svært eksponert | very exposed | ≥ 1 000 000, < 2 000 000 |
| 9 | h | ekstremt eksponert | extremely exposed | ≥ 2 000 000, < 4 000 000 |
| 10 | y | disruptivt eksponert | disruptively exposed | ≥ 4 000 000 |


In [2]:
waves.classify.CLASSES

,class_int,trinn,navn_no,navn_en,swm_lower,swm_upper
0,1,0,minimal vannforstyrrelsesintensitet,still water,NaN,1200.0
1,2,a,svært beskyttet,very sheltered,1200.0,4000.0
2,3,b,temmelig beskyttet,moderately sheltered,4000.0,10000.0
3,4,c,litt beskyttet,slightly sheltered,10000.0,50000.0
4,5,d,svakt eksponert,weakly sheltered,50000.0,100000.0
5,6,e,litt eksponert,slightly exposed,100000.0,500000.0
6,7,f,temmelig eksponert,moderately exposed,500000.0,1000000.0
7,8,g,svært eksponert,very exposed,1000000.0,2000000.0
8,9,h,ekstremt eksponert,extremely exposed,2000000.0,4000000.0
9,10,y,disruptivt eksponert,disruptively exposed,4000000.0,NaN


In [3]:
from pathlib import Path
ROOT_PATH = Path("..")
SRC  = ROOT_PATH / "niva/EswmRaster_filled_clipped_cog.tif"
CLF  = ROOT_PATH / "niva/EswmRaster_classified.tif"
SIE  = ROOT_PATH / "niva/EswmRaster_classified_sieved.tif"
VRAW  = ROOT_PATH / "niva/EswmRaster_classified_raw.gpkg"
VCLIP = ROOT_PATH / "niva/EswmRaster_classified_clipped.gpkg"
VOUT = ROOT_PATH / "niva/EswmRaster_classified_smooth.gpkg"
FIN = ROOT_PATH / "niva/EswmRaster_mapped_classified_smooth.gpkg"

In [ ]:
waves.classify.reclassify_raster(SRC, CLF)
print(f"Classified raster saved: {CLF}")

In [4]:
waves.classify.sieve_filter(CLF, SIE, threshold=2)
print(f"Sieved raster saved: {SIE}")

Copying classified raster...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:11.
Applying sieve filter (threshold=2, connectedness=8)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:15.
Sieved raster saved: ../niva/EswmRaster_classified_sieved.tif


In [5]:
waves.classify.polygonize(SIE, VRAW)
print(f"Raw polygons saved: {VRAW}")

Polygonizing (this may take a while)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:58.
Raw polygons saved: ../niva/EswmRaster_classified_raw.gpkg


In [8]:
import importlib

importlib.reload(waves.classify)

<module 'waves.classify' from '/home/kim/work/eswm-bolgemodell/waves/classify.py'>

In [9]:
AOI = "gs://niva-geodata/MarintNaturKart/aux/aoi_from_marine_vanntyper.geo.parquet"

waves.classify.clip_to_aoi(VRAW, VCLIP, AOI)


Warning 1: Ring Self-intersection at or near point 200865.90999602573 7103313.7599795749


RuntimeError: cannot load source clip geometry
May be caused by: Geometry of feature 7 of /home/kim/work/eswm-bolgemodell/niva/tmp_mv_outline.gpkg is invalid. You can try to make it valid by specifying -makevalid, but the results of the operation should be manually inspected.

In [ ]:
gdf = waves.classify.smooth_vectors(VCLIP, VOUT)
gdf.head()

Reading raw polygons...
Topology-aware simplification (tolerance=25.0 m)...


In [ ]:
cols = ["class_int", "trinn", "navn_no", "navn_en"]
gdf = gdf.merge(waves.classify.CLASSES[cols], on="class_int", how="left")
gdf.head()

,class_int,geometry,trinn_x,navn_no_x,navn_en_x,swm_lower,swm_upper,trinn_y,navn_no_y,navn_en_y
0,7,"POLYGON ((953883.819 7939945.624, 953874.444 7...",f,temmelig eksponert,moderately exposed,500000.0,1000000.0,f,temmelig eksponert,moderately exposed
1,6,"POLYGON ((953883.819 7939920.624, 953874.444 7...",e,litt eksponert,slightly exposed,100000.0,500000.0,e,litt eksponert,slightly exposed
2,7,"POLYGON ((973755.694 7939820.624, 973752.569 7...",f,temmelig eksponert,moderately exposed,500000.0,1000000.0,f,temmelig eksponert,moderately exposed
3,7,"POLYGON ((953541.631 7939784.686, 953541.631 7...",f,temmelig eksponert,moderately exposed,500000.0,1000000.0,f,temmelig eksponert,moderately exposed
4,6,"POLYGON ((953605.694 7939770.624, 953602.569 7...",e,litt eksponert,slightly exposed,100000.0,500000.0,e,litt eksponert,slightly exposed


In [ ]:
gdf.to_file(FIN, driver="GPKG")